# 34 · Tool annotations & safety hints

A tool can carry **annotations** that describe its behavior so a host can gate
it appropriately:

- `readOnlyHint` — does not modify state (auto-approve).
- `destructiveHint` — may delete/overwrite (require confirmation).
- `idempotentHint` — repeat calls are safe to retry.
- `openWorldHint` — touches external systems (network).

Both the from-scratch `MiniMCPServer` and the in-process `Session` support them.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # PHASE_4_MCP root
import mcp_lab
print('mcp_lab loaded; protocol', mcp_lab.PROTOCOL_VERSION)

In [ ]:
from mcp_lab import MiniMCPServer
s = MiniMCPServer('fs')
@s.tool(description='Read a file.', annotations={'readOnlyHint':True})
def read_file(path): return f'contents of {path}'
@s.tool(description='Delete a file.', annotations={'destructiveHint':True,'idempotentHint':False})
def delete_file(path): return f'deleted {path}'

for t in s.handle({'jsonrpc':'2.0','id':1,'method':'tools/list'})['result']['tools']:
    print(t['name'], '->', t.get('annotations'))

A host might auto-run `readOnlyHint` tools but pause for human approval on
anything `destructiveHint` — turning metadata into a safety policy.